# 03 · Build the training data for a two-stage curriculum

**The plan.** Teach the model Ekegusii first, then teach it public service
announcements — and keep a checkpoint after each phase so the two effects can be
told apart.

| Run | Trained on | What it is |
|---|---|---|
| **Stage 1** | Bible + storybooks + lughayangu | learns Ekegusii as a language. **This is the baseline.** |
| **Stage 2** | PSA_KE + 25% Bible replay, starting from stage 1 | adapts to PSA register. **This is the result.** |
| **Mixed** | everything at once, from scratch | control: would simply mixing have done as well? |

**Why the replay.** Stage 2 has only ~5,700 PSA rows against the ~58,000 the
model saw in stage 1. Fine-tuning on the small set alone causes catastrophic
forgetting — the model loses the general Ekegusii it just learned, and the Bible
test scores collapse. Mixing 25% of the stage-1 data back in anchors it.

**Why the mixed control.** Without it there is no evidence the *ordering*
mattered. If two-stage beats mixed, the curriculum is a finding; if it does not,
better to know before writing it up as one.

**Scope: Ekegusii only.** The 50k Kiswahili PSA corpus is not used. Directions
trained are `eng→guz` and `swh→guz`; Kiswahili appears only as a *source*
language, drawn from the Bible and PSA_KE. Notebook 02 is therefore optional.

**Inputs** — `bible_en_guz_swh.csv`, `lughayangu_sentences.csv`,
`psa_ke_train.csv`, `psa_ke_test.csv`
**Outputs** — `artifacts/data/{stage1,stage2,mixed,dev,test}.jsonl` + `mixture.json`
**Runtime** — under a minute.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
sys.path.insert(0, str(pathlib.Path.cwd()))
import nb_common as C

C.set_seed()
C.use_house_style()
print(f"project root: {C.ROOT}")

In [ ]:
import pandas as pd, numpy as np, json
from collections import Counter

C.require_files(C.BIBLE_CSV, C.PSA_KE_TRAIN_CSV, C.PSA_KE_TEST_CSV)
# lughayangu is in REPO_PATHS, so ask for it rather than silently doing
# without: a bare node that skips it trains a SMALLER stage 1 than the paper
# reports, and nothing would say so.
try:
    C.require_files(C.LUGHAYANGU_CSV)
except FileNotFoundError as exc:
    print(f"  lughayangu unavailable, continuing without it: {exc}")
HAS_LUGHAYANGU = C.LUGHAYANGU_CSV.exists()
print(f"  {'ok    ' if HAS_LUGHAYANGU else 'absent'} lughayangu_sentences.csv (optional)")

## 1. Configuration

`REPLAY_FRACTION` is the anti-forgetting knob. At 0.25, a quarter of stage 2 is
stage-1 data. Raise it if the Bible scores drop between stage 1 and stage 2;
lower it if the model is not adapting to PSA register.

In [ ]:
LUGHAYANGU_HELDOUT = 200   # of 316; the rest go into stage 1
BIBLE_TEST = 1500          # held-out verses, shared by both directions
MIN_WORDS, MAX_WORDS = 2, 60

PSA_KE_UPSAMPLE = 4        # PSA rows are scarce and exactly on-target
REPLAY_FRACTION = 0.25     # share of stage 2 drawn from stage-1 data

print(f"replay: {REPLAY_FRACTION:.0%} of stage 2 | PSA_KE upsampled {PSA_KE_UPSAMPLE}x")

## 2. Load the corpora

In [ ]:
def rec(src_lang, tgt_lang, src, tgt, corpus, domain=""):
    return {"src_lang": src_lang, "tgt_lang": tgt_lang,
            "src": str(src).strip(), "tgt": str(tgt).strip(),
            "corpus": corpus, "domain": domain}

def usable(r):
    """
    Length gate, applied to EVERY held-out row and to no PSA training row.

    That asymmetry is deliberate, not an oversight. The PSA corpus contains
    long document extracts and length-ratio outliers which were kept on
    purpose - they are still Ekegusii, and they still teach the language.
    Scoring on them is a different matter: a 200-word extract would dominate
    a corpus-level chrF and measure document handling rather than the PSA
    translation the paper is about. So: keep everything in train, hold the
    test set to 2-60 words on both sides.
    """
    ns, nt = len(r["src"].split()), len(r["tgt"].split())
    return (MIN_WORDS <= ns <= MAX_WORDS) and (MIN_WORDS <= nt <= MAX_WORDS)

bible = pd.read_csv(C.BIBLE_CSV)
bible_en_guz  = [rec(C.ENG, C.GUZ, r.english, r.ekegusii, "bible") for r in bible.itertuples()]
bible_swh_guz = [rec(C.SWH, C.GUZ, r.swahili, r.ekegusii, "bible") for r in bible.itertuples()]

if HAS_LUGHAYANGU:
    lg = pd.read_csv(C.LUGHAYANGU_CSV)
    lugha = [rec(C.ENG, C.GUZ, r.english, r.ekegusii, "lughayangu") for r in lg.itertuples()]
else:
    lugha = []

ke  = pd.read_csv(C.PSA_KE_TRAIN_CSV).fillna("")
ket = pd.read_csv(C.PSA_KE_TEST_CSV).fillna("")
ke_swh  = ke[ke["kiswahili"].astype(str).str.strip() != ""]
ket_swh = ket[ket["kiswahili"].astype(str).str.strip() != ""]

ke_en_guz  = [rec(C.ENG, C.GUZ, r.english,   r.ekegusii, "psa_ke", r.domain) for r in ke.itertuples()]
ke_swh_guz = [rec(C.SWH, C.GUZ, r.kiswahili, r.ekegusii, "psa_ke", r.domain) for r in ke_swh.itertuples()]

flagged = int((ke["quality_flags"].astype(str).str.strip() != "").sum())
print(f"bible        {len(bible):,} triples -> 2 directions")
print(f"lughayangu   {len(lugha):,} pairs")
print(f"PSA_KE       {len(ke):,} rows ({flagged:,} flagged non-PSA register, kept for")
print(f"             general language learning) -> en->guz {len(ke_en_guz):,},"
      f" swh->guz {len(ke_swh_guz):,}")

## 3. Splits, with the leakage traps closed

Two traps specific to this data:

1. **A verse appears in both directions.** If verse 500 is train for `en→guz`
   and test for `swh→guz`, its Ekegusii target has been seen. One split is made
   over verse indices and applied to both directions.
2. **PSA_KE was already split** by `prepare_psa_ke.py`, stratified by domain and
   with all flagged rows kept out of the test side. That split is reused rather
   than made again here.

In [ ]:
rng = np.random.default_rng(C.SEED)
n = len(bible)
perm = rng.permutation(n)
test_idx = set(perm[:BIBLE_TEST].tolist())
dev_idx  = set(perm[BIBLE_TEST:BIBLE_TEST + BIBLE_TEST // 2].tolist())

def bible_split(rows):
    tr, dv, te = [], [], []
    for i, r in enumerate(rows):
        if usable(r):
            (te if i in test_idx else dv if i in dev_idx else tr).append(r)
    return tr, dv, te

b1_tr, b1_dv, b1_te = bible_split(bible_en_guz)
b2_tr, b2_dv, b2_te = bible_split(bible_swh_guz)

lg_use = [r for r in lugha if usable(r)]
rng.shuffle(lg_use)
lg_te, lg_tr = lg_use[:LUGHAYANGU_HELDOUT], lg_use[LUGHAYANGU_HELDOUT:]
for r in lg_te:
    r["corpus"] = "lughayangu_contemporary"

ke_te = ([rec(C.ENG, C.GUZ, r.english,   r.ekegusii, "psa_ke_heldout", r.domain) for r in ket.itertuples()] +
         [rec(C.SWH, C.GUZ, r.kiswahili, r.ekegusii, "psa_ke_heldout", r.domain) for r in ket_swh.itertuples()])
ke_te = [r for r in ke_te if usable(r)]

print(f"bible  en->guz  train {len(b1_tr):,}  dev {len(b1_dv):,}  test {len(b1_te):,}")
print(f"bible  swh->guz train {len(b2_tr):,}  dev {len(b2_dv):,}  test {len(b2_te):,}")
print(f"lughayangu      train {len(lg_tr):,}  test {len(lg_te):,}")
print(f"PSA_KE                          test {len(ke_te):,}")

## 4. Assemble the three training sets

**What the mixed control does and does not control for.** `mixed` is
`stage1 + psa_up`, so it contains every *unique* example the two-stage run
ever sees — the replay slice is duplicated Bible material already present in
stage 1, not new data. What it does not match is the *number of gradient
updates*: at a fixed epoch count the two-stage run takes roughly 9% more
steps, because the replay rows are trained on twice. Report that alongside
the curriculum result. If the gap between stage 2 and mixed is small, the
budget difference is a live alternative explanation and the honest reading is
"no measurable curriculum benefit."

Upsampling is materialised here — the rows are physically repeated — so that
what the model sees is exactly what `mixture.json` reports. No hidden sampler.

In [ ]:
def dedupe(rows):
    seen, out = set(), []
    for r in rows:
        key = (r["src_lang"], r["tgt_lang"], r["src"].lower(), r["tgt"].lower())
        if key not in seen:
            seen.add(key); out.append(r)
    return out

# --- stage 1: general Ekegusii ------------------------------------------------
stage1 = dedupe(b1_tr + b2_tr + lg_tr)

# --- stage 2: PSA register, anchored by a replay slice of stage 1 -------------
psa_pool = dedupe(ke_en_guz + ke_swh_guz)
psa_up = psa_pool * PSA_KE_UPSAMPLE
n_replay = int(len(psa_up) * REPLAY_FRACTION / (1 - REPLAY_FRACTION))
replay_idx = rng.choice(len(stage1), size=min(n_replay, len(stage1)), replace=False)
replay = [dict(stage1[i], corpus="bible_replay") for i in replay_idx]
stage2 = psa_up + replay

# --- mixed control: everything at once ----------------------------------------
mixed = stage1 + psa_up

dev  = dedupe(b1_dv + b2_dv)
test = dedupe(b1_te + b2_te + lg_te + ke_te)

# The Bible contains 421 duplicate English verses - parallel passages where
# Chronicles repeats Kings, and so on. A verse can therefore be drawn into the
# test split while an identical copy sits in training, leaking the source. Drop
# those rows from test and dev rather than pretending the split was clean.
train_keys = {(r["src_lang"], r["tgt_lang"], r["src"].lower()) for r in stage1 + psa_pool}
def drop_leaks(rows, label):
    clean = [r for r in rows if (r["src_lang"], r["tgt_lang"], r["src"].lower()) not in train_keys]
    if len(clean) != len(rows):
        print(f"  dropped {len(rows) - len(clean)} leaked rows from {label}")
    return clean

test = drop_leaks(test, "test")
dev = drop_leaks(dev, "dev")
for rows, label in [(test, "test"), (dev, "dev")]:
    assert not [r for r in rows
                if (r["src_lang"], r["tgt_lang"], r["src"].lower()) in train_keys], \
        f"{label} still overlaps train after drop_leaks()"
print("leakage check passed (test and dev)\n")

print(f"stage 1  {len(stage1):>7,}   Bible + storybooks + lughayangu")
print(f"stage 2  {len(stage2):>7,}   {len(psa_up):,} PSA (x{PSA_KE_UPSAMPLE}) "
      f"+ {len(replay):,} replay ({100*len(replay)/len(stage2):.0f}%)")
print(f"mixed    {len(mixed):>7,}   stage 1 + PSA in one pass")
print(f"dev      {len(dev):>7,}")
print(f"test     {len(test):>7,}")

In [ ]:
def write_jsonl(rows, name):
    path = C.DATA / name
    with open(path, "w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  {len(rows):>7,} rows -> {path}")

for rows, name in [(stage1, "stage1.jsonl"), (stage2, "stage2.jsonl"),
                   (mixed, "mixed.jsonl"), (dev, "dev.jsonl"), (test, "test.jsonl")]:
    write_jsonl(rows, name)

## 5. What each run actually sees

In [ ]:
import matplotlib.pyplot as plt

runs = {"stage 1": stage1, "stage 2": stage2, "mixed": mixed}
corpora = ["bible", "bible_replay", "lughayangu", "psa_ke"]
counts = {r: Counter(x["corpus"] for x in rows) for r, rows in runs.items()}

fig, ax = plt.subplots(figsize=(9.5, 4))
bottom = np.zeros(len(runs))
xs = np.arange(len(runs))
for i, corpus in enumerate(corpora):
    vals = np.array([counts[r].get(corpus, 0) for r in runs])
    if vals.sum() == 0:
        continue
    ax.bar(xs, vals, 0.55, bottom=bottom, label=corpus, color=C.PALETTE[i],
           edgecolor="white", linewidth=2)
    bottom += vals
for i, r in enumerate(runs):
    ax.annotate(f"{len(runs[r]):,}", (i, bottom[i]), xytext=(0, 5),
                textcoords="offset points", ha="center", color=C.INK_MUTED)
ax.set_xticks(xs); ax.set_xticklabels(runs.keys())
ax.set_ylabel("training examples"); ax.set_title("Composition of each training run")
ax.legend()
C.save_fig(fig, "03_curriculum_mixture"); plt.show()

for r in runs:
    total = len(runs[r])
    parts = "  ".join(f"{k} {100*v/total:.0f}%" for k, v in counts[r].most_common())
    print(f"  {r:<9} {total:>7,}   {parts}")

In [ ]:
C.save_json({
    "scope": "ekegusii_only",
    "directions": ["eng_Latn->guz_Latn", "swh_Latn->guz_Latn"],
    "counts": {"stage1": len(stage1), "stage2": len(stage2), "mixed": len(mixed),
               "dev": len(dev), "test": len(test)},
    "psa_ke_upsample": PSA_KE_UPSAMPLE,
    "replay_fraction": REPLAY_FRACTION,
    "replay_rows": len(replay),
    "composition": {r: dict(counts[r]) for r in runs},
}, C.DATA / "mixture.json")
print("\nNext: 04_extend_tokenizer.ipynb, then 05_finetune.ipynb")